# 1. 환경설정

In [1]:
# 표준 라이브러리만 사용 (DB 연결 없이 실행 가능)
import math
from statistics import NormalDist

# 2. A/B 테스트 표본 수 계산

In [2]:
# 실험 설계 입력값
p_a = 0.0654          # 현재 리뷰 더보기 클릭 사용자의 콘텐츠 시작 전환율 (A: 최신순)
mde = 0.02            # 목표 개선폭 +2%p
p_b = p_a + mde       # 목표 전환율 (B: 추천수 기준 베스트순)
alpha = 0.05          # 유의수준 (양측)
power = 0.80          # 검정력
daily_visitors = 143  # 일 평균 리뷰 방문자 수

if not (0 < p_a < 1 and 0 < p_b < 1):
    raise ValueError("전환율은 0과 1 사이여야 합니다")

z_alpha = NormalDist().inv_cdf(1 - alpha / 2)
z_beta = NormalDist().inv_cdf(power)
p_bar = (p_a + p_b) / 2

# 두 비율 비교(양측 Z검정) 표본 수 공식
n_per_group = math.ceil(
    (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
     + z_beta * math.sqrt(p_a * (1 - p_a) + p_b * (1 - p_b))) ** 2
    / (p_b - p_a) ** 2
)
n_total = n_per_group * 2
days = math.ceil(n_total / daily_visitors)

print(f"전환율: {p_a:.2%} → {p_b:.2%} (+{mde*100:.0f}%p)")
print(f"그룹당 필요 표본: {n_per_group:,}명")
print(f"총 필요 표본: {n_total:,}명")
print(f"예상 실험 기간: 약 {days}일 (일 평균 {daily_visitors}명 기준)")

전환율: 6.54% → 8.54% (+2%p)
그룹당 필요 표본: 2,735명
총 필요 표본: 5,470명
예상 실험 기간: 약 39일 (일 평균 143명 기준)


In [3]:
# MDE별 필요 표본·기간 비교 (한 달 내외로 검증 가능한 개선폭 역산)
for d in [0.01, 0.015, 0.02, 0.025, 0.03]:
    pb = p_a + d
    pbar = (p_a + pb) / 2
    n = math.ceil(
        (z_alpha * math.sqrt(2 * pbar * (1 - pbar))
         + z_beta * math.sqrt(p_a * (1 - p_a) + pb * (1 - pb))) ** 2
        / (pb - p_a) ** 2
    )
    print(f"+{d*100:.1f}%p → 그룹당 {n:,}명 / 약 {math.ceil(n * 2 / daily_visitors)}일")

+1.0%p → 그룹당 10,273명 / 약 144일
+1.5%p → 그룹당 4,715명 / 약 66일
+2.0%p → 그룹당 2,735명 / 약 39일
+2.5%p → 그룹당 1,803명 / 약 26일
+3.0%p → 그룹당 1,289명 / 약 19일


💡 일 평균 리뷰 방문자가 약 143명이라 작은 전환율 차이는 검증에 장기간이 필요. 한 달 내외의 실험 기간을 고려해 목표 개선폭을 +2%p로 설정 → 그룹당 2,735명(총 5,470명), 약 39일

# 3. 기대 효과 추정

In [4]:
# 실험 미실행으로, 2023년 하반기 월 평균 리뷰 방문자에 목표치 +2%p를 적용해 추정
monthly_review_visitors = 4300  # 2023년 하반기 월 평균 리뷰 방문자
sub_rate = 0.0979               # 가입자 기준 월 구독 전환율
price_1m = 15920                # 1개월 플랜 가격(원)

add_start = round(monthly_review_visitors * mde)  # 월 추가 콘텐츠 시작
add_sub = round(add_start * sub_rate)             # 월 추가 구독자
add_rev = add_sub * price_1m                      # 월 추가 매출

print(f"월 추가 콘텐츠 시작: +{add_start}명")
print(f"월 추가 구독자: +{add_sub}명")
print(f"월 추가 매출: 약 {add_rev / 10000:.1f}만 원")
print(f"연 추가 매출: 약 {add_rev * 12 / 10000:.0f}만 원")

월 추가 콘텐츠 시작: +86명
월 추가 구독자: +8명
월 추가 매출: 약 12.7만 원
연 추가 매출: 약 153만 원


💡 월 +86명 → +8명 → 약 12.7만 원(연 약 153만 원). 기대 효과는 크지 않지만 추가 마케팅 비용 없이 적용할 수 있어 리뷰 노출 순서 변경을 시도할 가치가 있다고 판단